In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
ADLS_PATH =  'abfss://data@stdevfrancecentrallo1b.dfs.core.windows.net'

In [0]:
read_query = (
    spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('cloudFiles.schemaLocation', f'{ADLS_PATH}/schema')
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.maxFilesPerTrigger", 500)
    .load(f'{ADLS_PATH}/datasets')
    )

In [0]:
write_stream = (read_query.writeStream
                .format('delta')
                .outputMode('append')
                .trigger(availableNow=True)
                .option('checkpointLocation', '/Volumes/dbw_devfrancecentrallo1b/bronze/bronze_checkpoints')
                .toTable('bronze.raw_btcusdt_spot'))

In [0]:
write_stream.awaitTermination()